# Image classification

The first stage of the IFT processing is to generate a 4-class image classification: land, water, ice, and cloud. 


```
classification(false_color_image, land mask)
    clouds = cloud_mask(false_color_image) |> apply_landmask
    ice = preliminary_ice_mask(red band)
    water = not land, not ice, not cloud
    return masks
end
```
    



In [5]:
using Pkg
Pkg.activate("calval")
using CSVFiles
using DataFrames
using Dates
using IceFloeTracker
using Images
using Random
using StatsBase


  Activating project at `~/Documents/research/calval_tgrs/scripts/calval`


In [6]:
# load test images

function validated_binary_water(case::Case)
    info(case).water_sample == 0 && return nothing
    (; case_number, region, date, satellite) = IceFloeTracker.Data._filename_parts(case)
    file = "data/validation_dataset/binary_water_samples/$(case_number)-$(region)-$(date)-$(satellite)-binary_water_samples.png"
    img = file |> case.loader |> load |> binarize_mask .|> Gray
    return img
end

function validated_binary_cloud(case::Case)
    info(case).ice_sample == 0 && return nothing
    (; case_number, region, date, satellite) = IceFloeTracker.Data._filename_parts(case)
    file = "data/validation_dataset/binary_ice_samples/$(case_number)-$(region)-$(date)-$(satellite)-binary_ice_samples.png"
    img = file |> case.loader |> load |> binarize_mask .|> Gray
    return img
end

function validated_binary_ice(case::Case)
    info(case).cloud_sample == 0 && return nothing
    (; case_number, region, date, satellite) = IceFloeTracker.Data._filename_parts(case)
    file = "data/validation_dataset/binary_cloud_samples/$(case_number)-$(region)-$(date)-$(satellite)-binary_cloud_samples.png"
    img = file |> case.loader |> load |> binarize_mask .|> Gray
    return img
end

validated_binary_ice (generic function with 1 method)

In [ ]:
dataset = Watkins2026Dataset(ref="main")
# Add a column to indicate whether water samples have been generated yet
water_sample_files = readdir("../../ice_floe_validation_dataset/data/validation_dataset/binary_water_samples/")
water_sample_files = filter(r -> r != ".DS_Store", water_sample_files)
water_sample_cases = parse.(Int64, first.(split.(water_sample_files, "-")))
dataset.info[:, :water_sample] = (x -> x ∈ water_sample_cases).(dataset.info.case_number)
